# Published EACF with Mimir and exact-window calibration

This notebook exposes each stage of the Nielsen et al. (2022) repeating-pattern workflow. Mimir computes the Lomb--Scargle spectrum from observed timestamps; Urdr estimates the background, applies the Hanning filter bank, takes the complex inverse-transform magnitude, restricts the search to the physical $\nu_{\max}$--$\Delta\nu$ banana, and calibrates the global maximum with the same observing window.

In [ ]:
import asteroscale as ast
import matplotlib.pyplot as plt
import numpy as np
from mimir import power_spectrum, spectral_window
from urdr import (
    AsteroScaleSamples,
    EmpiricalBackgroundConfig,
    PublishedEACFSearch,
    SimulationConfig,
    calibrate_published_eacf,
    compute_published_eacf_map,
    estimate_empirical_background,
    make_observing_window,
    simulate_time_series,
)

## 1. A gapped test light curve

Missing cadences stay in Urdr's regular grid as a Boolean mask. Mimir receives only the genuinely observed timestamps, so the Lomb--Scargle calculation does not interpret a gap as zero flux.

In [ ]:
window = make_observing_window(
    duration_days=12.0,
    cadence_seconds=600.0,
    gaps_days=((4.0, 4.5), (8.0, 8.2)),
    random_missing_fraction=0.03,
    seed=10,
)
config = SimulationConfig(
    white_noise_sigma=0.25,
    granulation_amplitude=0.08,
    granulation_timescale_days=0.12,
    numax_uhz=500.0,
    delta_nu_uhz=32.0,
    envelope_width_uhz=160.0,
    oscillation_amplitude=0.5,
    mode_linewidth_uhz=2.0,
)
series = simulate_time_series(window, config, np.random.default_rng(20))
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(series.time, series.flux, lw=0.6)
ax.set(xlabel='Time [d]', ylabel='Flux', title=f'Observed duty cycle: {series.window.duty_cycle:.1%}')
plt.show()

## 2. Lomb--Scargle S/N spectrum and spectral window

The empirical running median follows the expected envelope-width scale. Dividing the power density by this estimate suppresses granulation without fitting a fixed number of Harvey components. The spectral window summarizes the same timestamp pattern that is reproduced explicitly in the null simulations.

In [ ]:
observed = series.observed
spectrum = power_spectrum(time=series.time[observed], flux=series.flux[observed])
background = estimate_empirical_background(
    spectrum.frequency, spectrum.power_density, EmpiricalBackgroundConfig()
)
sampling = spectral_window(time=series.time[observed], half_width=20.0)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].loglog(spectrum.frequency, spectrum.power_density, color='0.65', lw=0.6, label='Mimir PSD')
axes[0].loglog(spectrum.frequency, background, color='C1', lw=2, label='empirical background')
axes[0].set(xlabel=r'Frequency [$\mu$Hz]', ylabel='Power density')
axes[0].legend()
axes[1].plot(sampling.frequency, sampling.power)
axes[1].set(xlabel=r'Frequency offset [$\mu$Hz]', ylabel='Window power')
plt.show()

## 3. Smooth complex-modulus EACF map

For every trial centre, Urdr applies the published Hanning window to the S/N spectrum and computes $|\mathcal{F}^{-1}(S W)|^2$. The absolute value removes the rapid carrier oscillation. The dashed curves enclose the physically allowed scaling-relation region.

In [ ]:
centres = np.linspace(350.0, 650.0, 41)
eacf = compute_published_eacf_map(series, centres, max_lag_seconds=70_000.0)
expected_lag = 1e6 / eacf.predicted_delta_nu_uhz
factor = 10**0.2
fig, ax = plt.subplots(figsize=(9, 5))
image = ax.pcolormesh(centres, eacf.lags_seconds / 3600, eacf.values.T, shading='auto')
ax.plot(centres, expected_lag / factor / 3600, 'w--', lw=1.5)
ax.plot(centres, expected_lag * factor / 3600, 'w--', lw=1.5)
ax.scatter([config.numax_uhz], [1e6 / config.delta_nu_uhz / 3600], c='r', marker='x', s=80)
ax.set(xlabel=r'Trial filter centre [$\mu$Hz]', ylabel='Lag [h]')
fig.colorbar(image, ax=ax, label=r'Normalized $|C(\tau)|^2$')
plt.show()

## 4. Collapse the banana region

Each row is averaged only over its physically allowed lags. The maximum of this curve is the global statistic used for detection; retaining the maximum in every null realization includes the look-elsewhere effect.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(centres, eacf.collapsed())
ax.axvline(config.numax_uhz, color='C3', ls='--', label=r'injected $\nu_{\max}$')
ax.set(xlabel=r'Trial filter centre [$\mu$Hz]', ylabel='Mean physical-region EACF')
ax.legend()
plt.show()
print(f'best numax = {eacf.best_numax_uhz:.1f} uHz')
print(f'best delta_nu = {eacf.best_delta_nu_uhz:.1f} uHz')

## 5. AsteroScale-informed search

AsteroScale predicts a paired distribution of $\nu_{\max}$, $\Delta\nu$, and envelope width from external stellar information. Urdr converts those paired samples into local conditional search bands. This is narrower than the broad published banana, so it is reported as a separate, explicitly prior-informed result.

In [ ]:
asteroscale_output = ast.solve(
    given={
        'M': (1.0, 0.05),
        'R': (2.6, 0.10),
        'Teff': (5600.0, 80.0),
        'FeH': (0.0, 0.10),
    },
    want=['numax', 'dnu', 'FWHM_env'],
    preset='fast',
    seed=30,
)
asteroscale_samples = AsteroScaleSamples.from_mapping(asteroscale_output)
informed_search = PublishedEACFSearch.from_asteroscale(
    asteroscale_samples, centre_count=15, credible_mass=0.99
)
informed_eacf = compute_published_eacf_map(
    series, search=informed_search, max_lag_seconds=70_000.0
)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(centres, eacf.collapsed(), label='broad scaling search')
ax.plot(
    informed_eacf.centre_frequencies_uhz,
    informed_eacf.collapsed(),
    label='AsteroScale-informed search',
)
ax.set(xlabel=r'Trial filter centre [$\mu$Hz]', ylabel='Mean physical-region EACF')
ax.legend()
plt.show()

## 6. Target-specific detection and runtime controls

The short runs below are pedagogical. Scientific tail probabilities need enough null realizations to resolve the intended threshold. Null periodograms are processed in bounded batches, and SciPy can parallelise the inverse FFTs. `batch_size` trades memory for speed; `fft_workers=-1` uses every available CPU core. Every null still uses the target's exact timestamps and full Mimir-equivalent Lomb--Scargle/background/search pipeline.

In [ ]:
detection = calibrate_published_eacf(
    series,
    config,
    centres,
    simulations=16,
    seed=42,
    batch_size=16,
    fft_workers=2,
    max_lag_seconds=70_000.0,
)
informed_detection = calibrate_published_eacf(
    series,
    config,
    search=informed_search,
    simulations=16,
    seed=42,
    batch_size=16,
    fft_workers=2,
    max_lag_seconds=70_000.0,
)
print(f'broad:       p_FA={detection.false_alarm_probability:.3f}, merit={detection.detection_merit:.3f}')
print(f'AsteroScale: p_FA={informed_detection.false_alarm_probability:.3f}, merit={informed_detection.detection_merit:.3f}')